# Notebook 3: Import / Generate Your Data

In this notebook, we demonstrate several approaches for importing and generating data for hyperscanning analysis:

- **MATLAB Data:** Load data from a MATLAB file (e.g. .mat file) using `scipy.io.loadmat`.
- **EEG Data:** Import EEG data using MNE (from a .fif file).
- **XDF Data:** Import synchronized streams from XDF files using `mne_xdf`.
- **fNIRS Data:** Import fNIRS data (e.g., from a NIRx system) using MNE.
- **Simulation:** Generate synthetic data using coupled oscillator models to create controlled synchrony between participants.

In [ ]:
import os
import numpy as np
import mne
import pickle  # Added for loading saved data
from scipy.io import loadmat
import matplotlib.pyplot as plt
from pathlib import Path  # Added for path handling
from copy import copy
from collections import OrderedDict

import pyxdf  # Use pyxdf for XDF loading

# HyPyP modules
import hypyp.analyses as analyses
from hypyp.fnirs_tools import load_fnirs
from hypyp.fnirs_tools import make_fnirs_montage
from hypyp.fnirs_tools import fnirs_epoch
from hypyp import prep  # requires autoreject installation
from hypyp import stats
from hypyp import viz
from hypyp import utils  # Added for the simulation part

print("Libraries imported successfully.")

## 1. Importing MATLAB Data

Here we load MATLAB data from a .mat file using `scipy.io.loadmat`.
Assume that `data_matlab.mat` contains a variable called 'data' (for example, EEG signals or similar).

In [ ]:
# Path to your MATLAB file
filename = "pce01_P1_Rest1"
matlab_rest_file = os.path.join("./data/pce01230807/eeg_recordings/", f"{filename}.mat")

# Load the .mat file
try:
    mat_rest = loadmat(matlab_rest_file)
    print(f"Loaded trial file: {matlab_rest_file}. Available keys: {list(mat_rest.keys())}")

    rest_key = "pce_P1_Rest"
    data_matlab = mat_rest.get(rest_key)
    if data_matlab is None:
        raise KeyError(f"Variable {rest_key} not found in the MATLAB file.")
    print("MATLAB data loaded successfully. Data shape:", data_matlab.shape)
except Exception as e:
    print("Error loading MATLAB data:", e)

sfreq_rest = 1000
print(f"Resting state data shape: {data_matlab.shape}, SFreq: {sfreq_rest} Hz")

n_channels_rest, n_samples_rest = data_matlab.shape
ch_names = [f'EEG{i+1}' for i in range(n_channels_rest)]
data_rest = data_matlab.astype(np.float32)
info_rest = mne.create_info(ch_names=ch_names, sfreq=sfreq_rest, ch_types=['eeg'] * n_channels_rest)
raw_rest = mne.io.RawArray(data_rest, info_rest)

try:
    montage_std = mne.channels.make_standard_montage('standard_1020')
    raw_rest.set_montage(montage_std, on_missing='ignore')
    print("Standard montage applied to resting state as fallback.")
except Exception as e:
    print("Error setting standard montage for resting state:", e)


fname = os.path.join("./data/pce01230807/", f"{filename}_raw.fif")
raw_rest.save(fname, overwrite=True)

## 2. Importing EEG Data using MNE

We load EEG data from a .fif file using MNE.  
Make sure your file (e.g., `eeg_data.fif`) contains raw EEG recordings.

In [ ]:
# Path to your EEG .fif file
filename = "pce01_P1_Rest1"
eeg_file = os.path.join("./data/pce01230807/", f"{filename}_raw.fif")

try:
    raw_eeg = mne.io.read_raw_fif(eeg_file, preload=True)
    # Optionally, set a montage if not already set (e.g., standard 10-20 montage)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw_eeg.set_montage(montage)
    print("EEG data loaded successfully. Info:")
    print(raw_eeg.info)
    
    # Plot a short segment of the raw data
    raw_eeg.plot(n_channels=8, duration=5, title="Raw EEG Data")
except Exception as e:
    print("Error loading EEG data:", e)

## 3 Importing XDF Data

XDF (eXtensible Data Format) is a file format for storing synchronized streams of time series data. It's commonly used in Lab Streaming Layer (LSL) recordings, which are popular in hyperscanning experiments as they allow for easy synchronization of data across multiple devices and participants.

Below we demonstrate how to load an XDF file containing EEG data from two participants recorded simultaneously.

In [ ]:
# Path to XDF files
xdf_file = "./data/data_with_clock_resets.xdf"

try:
    # Load XDF files using pyxdf
    streams, header = pyxdf.load_xdf(xdf_file)
        
    print("XDF files loaded successfully.")
    print(f"Found {len(streams)} streams")
    
    # Print information about each stream to help identify what we're working with
    for i, stream in enumerate(streams):
        print(f"Stream {i}:")
        print(f"  Name: {stream.get('info', {}).get('name', ['Unknown'])[0]}")
        print(f"  Type: {stream.get('info', {}).get('type', ['Unknown'])[0]}")
        print(f"  Channel count: {stream.get('info', {}).get('channel_count', ['Unknown'])[0]}")
        print(f"  Sample rate: {stream.get('info', {}).get('nominal_srate', ['Unknown'])[0]}")
        print(f"  Data shape: {np.array(stream['time_series']).shape if 'time_series' in stream else 'Unknown'}")
        print(f"  First few values: {np.array(stream['time_series'])[:5] if 'time_series' in stream else 'Unknown'}")
    
    # Find which stream contains the EEG data
    eeg_stream_index = None
    for i, stream in enumerate(streams):
        if stream.get('info', {}).get('type', [''])[0] == 'EEG':
            eeg_stream_index = i
            break
    
    if eeg_stream_index is None:
        raise ValueError("No EEG stream found in the XDF file")
        
    # Process the EEG stream
    eeg_stream = streams[eeg_stream_index]
    data = np.array(eeg_stream["time_series"])
    print(f"EEG data shape: {data.shape}")
    
    # Get sampling frequency
    sfreq = float(eeg_stream["info"]["nominal_srate"][0])
    if sfreq <= 0:
        print("Warning: Sampling rate is not positive. Using a default value of 100 Hz.")
        sfreq = 100.0
    
    # Create channel names - simplified approach that works without desc info
    n_channels = int(eeg_stream["info"]["channel_count"][0])
    ch_names = [f'EEG{i+1}' for i in range(n_channels)]
    print(f"Using generic channel names: {ch_names}")
    
    # Create MNE info object
    ch_types = ['eeg'] * len(ch_names)
    info = mne.create_info(ch_names, sfreq, ch_types)
    
    # Create Raw object - transpose the data if necessary
    if data.shape[0] > data.shape[1]:  # If rows > columns (samples > channels)
        # Data format is samples x channels, needs to be channels x samples for MNE
        raw_xdf = mne.io.RawArray(data.T, info)
    else:
        # Data is already in channels x samples format
        raw_xdf = mne.io.RawArray(data, info)
    
    print(f"Created Raw object with {len(ch_names)} channels at {sfreq} Hz")
    
    # Look for scaling issues - check the mean and standard deviation
    data_mean = np.mean(np.abs(raw_xdf.get_data()))
    data_std = np.std(raw_xdf.get_data())
    print(f"Data absolute mean: {data_mean}, standard deviation: {data_std}")
    
    # Optionally, set a montage
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw_xdf.set_montage(montage, on_missing='ignore')
        print("Standard montage applied to XDF data.")
    except Exception as e:
        print(f"Warning: Could not set montage: {e}")
    
    # Plot a short segment of the raw data using MNE's built-in plotting
    raw_xdf.plot(n_channels=min(8, len(ch_names)), duration=5, title="Filtered EEG Data", scalings='auto')
    
    # Optionally, save as FIF for easier handling with MNE later
    raw_xdf.save("./data/subject_from_xdf_raw.fif", overwrite=True)
    print("XDF data converted to FIF files for future use.")
    
except FileNotFoundError:
    print("XDF file not found. This section requires an XDF file to run.")
except Exception as e:
    print(f"Error loading or processing XDF data: {e}")
    import traceback
    traceback.print_exc()

## 4. Importing fNIRS Data

MNE provides readers for fNIRS data, such as data from NIRx systems.
Below, we assume that the fNIRS data is stored in a folder (e.g., `nirx_data/`) as exported from a NIRx device.

In [ ]:
# Define file paths for the SNIRF data for two participants
path_1 = "./data/FNIRS/DCARE_02_sub1.snirf"
path_2 = "./data/FNIRS/DCARE_02_sub2.snirf"

print('Data paths set:')
print('Participant 1:', path_1)
print('Participant 2:', path_2)

In [ ]:
# Load fNIRS data for two participants
fnirs_data = load_fnirs(path_1, path_2, attr=None, preload=False, verbose=None)
fnirs_participant_1 = fnirs_data[0]
fnirs_participant_2 = fnirs_data[1]

print('fNIRS data loaded for both participants.')

In [ ]:
# Define source and detector labels
source_labels = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8']
detector_labels = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8']

# Path to the probe information file
prob_mat_file = './data/FNIRS/MCARE_01_probeInfo.mat'

# 3D coordinates (in mm) of anatomical landmarks
Nz_coord = [12.62, 17.33, 16.74]    # Tip of the nose
RPA = [21.0121020904262, 15.9632489747085, 17.2796094659563]    # Right preauricular
LPA = [4.55522116441745, 14.6744377188919, 18.3544292678269]     # Left preauricular

# Head size in mm
head_size = 0.16

# Create the montage using the provided probe information
location = make_fnirs_montage(source_labels, detector_labels, prob_mat_file,
                              Nz_coord, RPA, LPA, head_size)

print('Compatible montage created.')

In [ ]:
# Create Epoch objects for both participants
fnirs_epochs = fnirs_epoch(fnirs_participant_1, fnirs_participant_2,
                           tmin=-0.1, tmax=1, baseline=(None, 0),
                           preload=True, event_repeated='merge')

fnirs_epo1 = fnirs_epochs[0]
fnirs_epo2 = fnirs_epochs[1]

print('Epoch objects created for both participants.')

In [ ]:
# Set the montage for both epoch objects
fnirs_epo1.set_montage(location)
fnirs_epo2.set_montage(location)

print('Montage applied to both participants.')

In [ ]:
# Plot sensor locations with channel names for participant 1
fnirs_epo1.plot_sensors(show_names=True)

# Plot sensor locations with channel names for participant 2
fnirs_epo2.plot_sensors(show_names=True)

print('Sensor plots displayed.')

In [ ]:
# Plot the fNIRS data for participant 1
fnirs_epo1.plot()

# Plot the fNIRS data for participant 2
fnirs_epo2.plot()

print('fNIRS data plots displayed.')

## 5. Simulating Data with Coupled Oscillators

Instead of using MNE's source simulation tools, we'll demonstrate how to create simulated EEG data using a coupled oscillator approach. This method is particularly useful for hyperscanning as it allows us to control the level of synchronization between simulated brain signals of two participants.

The simulation follows these steps:
1. Set parameters for oscillator frequency and coupling
2. Define a coupling matrix to specify which channels are connected
3. Simulate phase evolution using differential equations
4. Convert phases to EEG-like signals
5. Package the simulated data into MNE Epochs objects

In [ ]:
# Define the frequency bands for analysis
freq_bands = {
    'Theta': [4, 7],
    'Alpha-Low': [7.5, 11],
    'Alpha-High': [11.5, 13],
    'Beta': [13.5, 29.5],
    'Gamma': [30, 48]
}

# Preserve the order using an OrderedDict
freq_bands = OrderedDict(freq_bands)

print('Frequency bands set:', freq_bands)

# Try to load epochs from previous notebooks first
try:
    with open('./data/hyperscanning_data.pkl', 'rb') as f:
        saved_data = pickle.load(f)
    
    # Extract the cleaned epochs from the loaded data
    epo1 = saved_data['epo1_clean']
    epo2 = saved_data['epo2_clean']
    print("Successfully loaded preprocessed epochs from previous notebook.")
except:
    # If that fails, load from files directly
    try:
        epo1 = mne.read_epochs("./data/participant1-epo.fif", preload=True)
        epo2 = mne.read_epochs("./data/participant2-epo.fif", preload=True)
        
        # Equalize the number of epochs between participants
        mne.epochs.equalize_epoch_counts([epo1, epo2])
        print("Loaded epochs from files.")
    except:
        print("No epoch data found. Creating mock data for simulation...")
        # Create mock data if no real data is found
        info = mne.create_info(ch_names=['Fz', 'Cz', 'Pz', 'Oz', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4'], 
                               sfreq=250, ch_types='eeg')
        data = np.random.randn(10, 10, 500)  # 10 channels, 10 epochs, 500 time points
        epo1 = mne.EpochsArray(data, info)
        epo2 = mne.EpochsArray(data, info)
        print("Created mock data for simulation.")

# Merge epochs from both participants
epo_real = utils.merge(epoch_S1=epo1, epoch_S2=epo2)

# Generate random epochs with a specified mean and standard deviation
epo_rnd = utils.generate_random_epoch(epoch=epo_real, mu=0.0, sigma=2.0)

# Get data shape and sampling frequency
n_epo, n_chan, n_samp = epo_real.get_data(copy=False).shape
sfreq = epo_real.info['sfreq']

print(f'epo_real shape: {n_epo} epochs, {n_chan} channels, {n_samp} samples per epoch')
print(f'Sampling rate: {sfreq} Hz')

In [ ]:
# Oscillator parameters
frequency_mean = 10.0  # Hz
frequency_std = 0.2    # Hz

noise_phase_level = 0.005 / n_samp
noise_amplitude_level = 0.0

# Determine the number of channels per group
N = int(n_chan / 2)

# Build the coupling matrix with two groups
A11 = 1 * np.ones((N, N))
A12 = 0 * np.ones((N, N))
A21 = 0 * np.ones((N, N))
A22 = 1 * np.ones((N, N))
W = np.block([[A11, A12], [A21, A22]])
W = 0.2 * W

# Visualize the coupling matrix - fixed version
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.matshow(W)
plt.title('Coupling Matrix W')
plt.colorbar(im)
plt.show()

print('Coupling matrix constructed.')

In [ ]:
# Total number of time points across all epochs
Nt = n_samp * n_epo

# Total duration in seconds
tmax = (n_samp / sfreq) * n_epo

# Create a time vector from 0 to tmax
tv = np.linspace(0., tmax, Nt)

# Generate random frequencies for each channel
freq = frequency_mean + frequency_std * np.random.randn(n_chan)
omega = 2. * np.pi * freq

def fp(p, t):
    """
    Compute the derivative of the phase vector for the coupled oscillators.
    
    Parameters:
        p (array): Phase vector (in radians).
        t (float): Time (not used explicitly here as the system is autonomous).
    
    Returns:
        array: Time derivative of phase vector.
    """
    p = np.atleast_2d(p)
    # Compute coupling term
    coupling = np.squeeze((np.sin(p) * np.matmul(W, np.cos(p).T).T) - (np.cos(p) * np.matmul(W, np.sin(p).T).T))
    # Compute phase derivative
    dotp = omega - coupling + noise_phase_level * np.random.randn(n_chan)
    return dotp

# Integrate the differential equation to obtain phase evolution
from scipy.integrate import odeint
p0 = 2 * np.pi * np.block([np.zeros(N), np.zeros(N) + np.random.rand(N) + 0.5])
Phi = odeint(fp, p0, tv) % (2 * np.pi)

# Plot phase evolution for the first epoch and the last epoch
plt.figure(figsize=(15, 8))
plt.subplot(2, 1, 1)
plt.imshow(Phi[:n_samp, :].T, interpolation='none', cmap='hsv')
plt.title('Phase evolution (first epoch)')
plt.colorbar(label='Phase (rad)')

plt.subplot(2, 1, 2)
plt.imshow(Phi[-n_samp:, :].T, interpolation='none', cmap='hsv')
plt.title('Phase evolution (last epoch)')
plt.colorbar(label='Phase (rad)')
plt.tight_layout()
plt.show()

print('Phase simulation completed.')

In [ ]:
# Plot signals for comparison
plt.figure(figsize=(15, 10))

# Plot original EEG (first epoch from merged real data)
plt.subplot(2, 2, 1)
plt.plot(tv[:n_samp], np.squeeze(epo_real[0]._data.T), "-")
plt.xlabel("Time, t [s]")
plt.ylabel("Amplitude")
plt.title('Original EEG')

# Plot random signal (first epoch from random data)
plt.subplot(2, 2, 2)
plt.plot(tv[:n_samp], np.squeeze(epo_rnd[0]._data).T, "-")
plt.xlabel("Time, t [s]")
plt.ylabel("Amplitude")
plt.title('Random signal')

# Plot sine of phase at start of simulation
plt.subplot(2, 2, 3)
plt.plot(tv[:n_samp], np.sin(Phi[:n_samp, :]), "-")
plt.xlabel("Time, t [s]")
plt.ylabel("Amplitude")
plt.title('Start of the simulation')

# Plot sine of phase at end of simulation
plt.subplot(2, 2, 4)
plt.plot(tv[-n_samp:], np.sin(Phi[-n_samp:, :]), "-")
plt.xlabel("Time, t [s]")
plt.ylabel("Amplitude")
plt.title('End of the simulation')

plt.tight_layout()
plt.show()

print('Signal plots generated.')

In [ ]:
def virtual_dyad(epochs=epo_real, frequency_mean=10., frequency_std=0.2, noise_phase_level=0.005, noise_amplitude_level=0.1, W=W):
    """
    Generate a simulated (virtual) dyad of EEG epochs based on coupled oscillator dynamics.
    
    Parameters:
        epochs (Epochs): Real EEG epochs to base the simulation on.
        frequency_mean (float): Mean oscillator frequency (Hz).
        frequency_std (float): Standard deviation of oscillator frequencies (Hz).
        noise_phase_level (float): Noise level added to phase dynamics.
        noise_amplitude_level (float): Amplitude noise level added to the simulated EEG.
        W (ndarray): Coupling matrix between channels.
    
    Returns:
        Epochs: Simulated EEG epochs with the same structure as the input epochs.
    """
    n_epo, n_chan, n_samp = epochs.get_data(copy=False).shape
    sfreq = epochs.info['sfreq']

    Nt = n_samp * n_epo
    tmax = (n_samp / sfreq) * n_epo  # total duration in seconds
    tv = np.linspace(0., tmax, Nt)

    # Generate random oscillator frequencies for each channel
    freq = frequency_mean + frequency_std * np.random.randn(n_chan)
    omega = 2. * np.pi * freq

    def fp(p, t):
        p = np.atleast_2d(p)
        coupling = np.squeeze((np.sin(p) * np.matmul(W, np.cos(p).T).T) - (np.cos(p) * np.matmul(W, np.sin(p).T).T))
        dotp = omega - coupling + noise_phase_level * np.random.randn(n_chan) / n_samp
        return dotp

    # Initial phases
    p0 = 2 * np.pi * np.block([np.zeros(int(n_chan/2)), np.zeros(int(n_chan/2)) + np.random.rand(int(n_chan/2)) + 0.5])

    # Integrate phase dynamics
    phi = odeint(fp, p0, tv) % (2 * np.pi)
    
    # Create simulated EEG signals as sine of the phases plus amplitude noise
    eeg = np.sin(phi) + noise_amplitude_level * np.random.randn(*phi.shape)
    
    # Reshape the simulated data to match the original epochs shape
    simulation = epochs.copy()
    simulation._data = np.transpose(np.reshape(eeg.T, [n_chan, n_epo, n_samp]), (1, 0, 2))
    
    return simulation

print('virtual_dyad function defined.')

In [ ]:
# Generate simulated epochs based on the real epochs
sim = virtual_dyad(epochs=epo_real, frequency_mean=10., frequency_std=0.2, 
                  noise_phase_level=0.005, noise_amplitude_level=0.1, W=W)

# Plot a few epochs from the simulated data
sim.plot(scalings=5, n_epochs=3, n_channels=min(62, n_chan))  # limit to 62 channels in case there are fewer
plt.show()

print('Simulated EEG (virtual dyad) plotted.')

# Optionally, save the simulated data
sim_data = {
    'simulated_epochs': sim,
    'coupling_matrix': W,
    'phase_evolution': Phi
}

with open('./data/simulated_dyad.pkl', 'wb') as f:
    pickle.dump(sim_data, f)
    
print("Simulated data saved to './data/simulated_dyad.pkl'")

## Summary

In this notebook, we have demonstrated multiple methods to import and generate data for hyperscanning analyses:

- **MATLAB Data:** Loaded using `scipy.io.loadmat`.
- **EEG Data:** Loaded from a .fif file using MNE.
- **XDF Data:** Imported from XDF files using the mne_xdf extension.
- **fNIRS Data:** Imported from a NIRx folder using MNE.
- **Simulation:** Generated simulated dyads using coupled oscillator models to create controlled synchrony between participants.

These techniques provide a solid foundation for subsequent analyses in your hyperscanning workflow. The simulated data is particularly useful for validating analysis methods, as it allows you to know the "ground truth" level of synchronization between the signals.